# Day 26: Agentic Workflows – Tool‑Using Agent with LangGraph

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, List
import operator

## 1. Define tools

In [ ]:
@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        result = eval(expression)
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {e}"

search = TavilySearchResults(max_results=2)

tools = [calculator, search]
tools_by_name = {tool.name: tool for tool in tools}

## 2. Define agent state

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[List[dict], operator.add]
    next_action: str

## 3. Build LangGraph agent

In [ ]:
llm = ChatOpenAI(model="gpt-4o", temperature=0)
llm_with_tools = llm.bind_tools(tools)

def agent_node(state: AgentState):
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

def should_continue(state: AgentState):
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return END

def tool_node(state: AgentState):
    last_message = state["messages"][-1]
    outputs = []
    for tool_call in last_message.tool_calls:
        tool = tools_by_name[tool_call["name"]]
        result = tool.invoke(tool_call["args"])
        outputs.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call["id"]
        })
    return {"messages": outputs}

workflow = StateGraph(AgentState)
workflow.add_node("agent", agent_node)
workflow.add_node("tools", tool_node)
workflow.set_entry_point("agent")
workflow.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
workflow.add_edge("tools", "agent")
app = workflow.compile()

## 4. Run the agent

In [ ]:
def ask_agent(question):
    inputs = {"messages": [{"role": "user", "content": question}]}
    final_state = None
    for output in app.stream(inputs):
        final_state = output
    # Extract final answer
    messages = final_state.get("agent", {}).get("messages", [])
    if messages:
        return messages[-1].content
    return "No answer"

response = ask_agent("What is 25 * 4 + 10? Also, who won the World Cup in 2018?")
print(response)